In [ ]:
print('kernel probe ok')

# 02 Configure

This notebook loads env/.env and validates both direct Azure OpenAI calls and APIM gateway calls.

## Load Environment

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path('../env/.env')
if not env_path.exists():
    raise RuntimeError('env/.env not found. Run 01_deploy_infra.ipynb first.')

load_dotenv(env_path)

AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT', '').rstrip('/')
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY', '')
AZURE_OPENAI_DEPLOYMENT = os.getenv('AZURE_OPENAI_DEPLOYMENT', '')
AZURE_OPENAI_SECONDARY_DEPLOYMENT = os.getenv('AZURE_OPENAI_SECONDARY_DEPLOYMENT', '')
AZURE_OPENAI_API_VERSION = os.getenv('AZURE_OPENAI_API_VERSION', '2024-10-21')
APIM_ENDPOINT = os.getenv('APIM_ENDPOINT', '').rstrip('/')
APIM_PATH = os.getenv('APIM_CHAT_COMPLETIONS_PATH', '/openai/chat/completions')
APIM_API_KEY = os.getenv('APIM_API_KEY', '')

print(f'Azure OpenAI endpoint: {AZURE_OPENAI_ENDPOINT}')
print(f'Primary deployment: {AZURE_OPENAI_DEPLOYMENT}')
print(f'Secondary deployment: {AZURE_OPENAI_SECONDARY_DEPLOYMENT}')
print(f'APIM endpoint: {APIM_ENDPOINT}')
print(f'APIM path: {APIM_PATH}')

## Test Direct Azure OpenAI (Primary + Secondary)

In [ ]:
import requests

def call_aoai(deployment_name: str, prompt: str):
    url = f"{AZURE_OPENAI_ENDPOINT}/openai/deployments/{deployment_name}/chat/completions?api-version={AZURE_OPENAI_API_VERSION}"
    headers = {
        'api-key': AZURE_OPENAI_API_KEY,
        'Content-Type': 'application/json'
    }
    payload = {
        'messages': [
            {'role': 'system', 'content': 'You are a concise assistant.'},
            {'role': 'user', 'content': prompt}
        ],
        'max_completion_tokens': 80
    }
    return requests.post(url, headers=headers, json=payload, timeout=45)

for dep in [AZURE_OPENAI_DEPLOYMENT, AZURE_OPENAI_SECONDARY_DEPLOYMENT]:
    print(f'Calling deployment: {dep}')
    resp = call_aoai(dep, 'Reply with: direct endpoint works')
    if resp.status_code == 200:
        data = resp.json()
        text = data.get('choices', [{}])[0].get('message', {}).get('content', '')
        print(f'  Success: {text}')
    else:
        print(f'  Failed: {resp.status_code}')
        print(resp.text[:400])

## Test APIM Gateway Endpoint

In [ ]:
import requests

if not APIM_API_KEY or APIM_API_KEY.startswith('[Retrieve from Azure Portal'):
    raise RuntimeError('APIM_API_KEY is not set with a real key in env/.env')

url = f"{APIM_ENDPOINT}{APIM_PATH}"
headers = {
    'Ocp-Apim-Subscription-Key': APIM_API_KEY,
    'Content-Type': 'application/json'
}
payload = {
    'messages': [
        {'role': 'system', 'content': 'You are a concise assistant.'},
        {'role': 'user', 'content': 'Reply with: apim endpoint works'}
    ],
    'max_completion_tokens': 80
}

response = requests.post(url, headers=headers, json=payload, timeout=45)
print(f'Status: {response.status_code}')
if response.status_code != 200:
    print(response.text[:600])
    raise RuntimeError('APIM test failed.')

body = response.json()
print(body.get('choices', [{}])[0].get('message', {}).get('content', ''))
print('APIM gateway validation succeeded')